# Detección de Fraude Financiero con Machine Learning
### Proyecto Final – Minería de Datos · Metodología CRISP-DM

**Dataset:** Credit Card Fraud Detection (Kaggle · `mlg-ulb/creditcardfraud`), archivo `creditcard.csv`.

Este notebook parte del ejemplo de *Stacking Ensemble* entregado en clase y lo amplía para cumplir todos los requerimientos del proyecto #11:

| Requerimiento | Dónde se cumple |
|---|---|
| Balanceo de clases (SMOTE) | Fase 3 |
| Random Forest (Bagging) | Fase 4.2 |
| XGBoost (Boosting), con y sin SMOTE | Fases 4.3 y 4.4 |
| Stacking (idea del ejemplo original) | Fase 4.5 |
| Red neuronal profunda (DNN) | Fase 4.6 |
| Autoencoder para anomalías | Fase 4.7 |
| Precision, Recall, F1, ROC-AUC (+ PR-AUC) | Fase 5 |
| Base para dashboard y alertas en tiempo real | Fase 6 |

**Principales mejoras frente al ejemplo original**

1. Se usa el dataset que exige el enunciado.
2. Se eliminan duplicados **antes** de dividir, para que una misma transacción no quede a la vez en entrenamiento y prueba.
3. División en tres conjuntos (entrenamiento / validación / prueba). El conjunto de prueba **no se toca** hasta la evaluación final; en el ejemplo original se usaba `X_test` dentro de `eval_set` durante el entrenamiento.
4. SMOTE se aplica **solo** al conjunto de entrenamiento.
5. El umbral de decisión se elige con datos de validación, no de forma arbitraria (el ejemplo usaba 0,80 sin justificar).
6. Se reportan métricas adecuadas para datos desbalanceados (PR-AUC, ROC-AUC, F1) y un análisis de costo económico, en lugar de basarse en el *accuracy*.
7. Se guardan **todos** los artefactos de preprocesamiento (scaler, lista de variables, umbrales) para que la predicción en producción sea idéntica a la del entrenamiento.

---
# Fase 1 · Comprensión del negocio

**Problema.** El uso no autorizado de tarjetas genera pérdidas millonarias a bancos, comercios y clientes. La entidad necesita identificar las transacciones fraudulentas en tiempo (casi) real para bloquearlas antes de que se complete el pago, sin afectar a los clientes legítimos.

**Objetivo de negocio.** Reducir la pérdida económica por fraude manteniendo un volumen manejable de alertas para el equipo de revisión.

**Objetivo de minería de datos.** Construir un clasificador binario que estime la probabilidad de que una transacción sea fraude (`Class = 1`).

**Costos de los errores**

| Error | Significado | Costo |
|---|---|---|
| Falso negativo (FN) | Fraude no detectado | Se pierde el monto de la transacción |
| Falso positivo (FP) | Transacción legítima bloqueada | Costo de revisión + molestia al cliente |

Un falso negativo suele costar mucho más que un falso positivo, por eso se prioriza el **Recall** sin descuidar la **Precision**.

**Criterios de éxito (en el conjunto de prueba)**
- PR-AUC (Average Precision) ≥ 0,80 — métrica principal por el fuerte desbalance.
- Recall ≥ 0,80 con Precision ≥ 0,70.
- Menor costo económico total que un sistema sin modelo.

---
# Configuración del entorno

In [ ]:
# Si trabajas en Google Colab o en un entorno nuevo, descomenta la siguiente línea:
# !pip install xgboost imbalanced-learn tensorflow shap kagglehub

In [ ]:
# =====================================================
# LIBRERÍAS
# =====================================================
import os
import json
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, ConfusionMatrixDisplay
)

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")

# =====================================================
# CONFIGURACIÓN GENERAL
# =====================================================
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# Ruta del dataset (en Colab sería, por ejemplo, "/content/creditcard.csv")
DATA_PATH = "data/raw/creditcard.csv"

MODELS_DIR = "models"
FIG_DIR = "reports/figures"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

def guardar_figura(nombre):
    # Guarda la figura actual en reports/figures para usarla en el artículo IEEE
    plt.savefig(os.path.join(FIG_DIR, nombre), bbox_inches="tight", dpi=150)

---
# Fase 2 · Comprensión de los datos

El dataset contiene transacciones realizadas con tarjetas europeas durante dos días de septiembre de 2013.

- `Time`: segundos transcurridos desde la primera transacción del dataset.
- `V1`–`V28`: componentes principales (PCA) de las variables originales, anonimizadas por confidencialidad.
- `Amount`: monto de la transacción (euros).
- `Class`: 1 = fraude, 0 = legítima.

In [ ]:
# =====================================================
# CARGAR DATASET
# =====================================================
# Si el archivo no está en DATA_PATH, se intenta descargar con kagglehub
if not os.path.exists(DATA_PATH):
    try:
        import kagglehub
        carpeta = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
        DATA_PATH = os.path.join(carpeta, "creditcard.csv")
    except Exception as e:
        raise FileNotFoundError(
            f"No se encontró {DATA_PATH}. Descarga creditcard.csv desde "
            "https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud y colócalo en esa ruta."
        ) from e

df = pd.read_csv(DATA_PATH)
print("Dimensiones:", df.shape)
df.head()

In [ ]:
# =====================================================
# INFORMACIÓN GENERAL Y CALIDAD DE DATOS
# =====================================================
df.info()

print("\nValores nulos totales:", int(df.isnull().sum().sum()))
print("Filas duplicadas:", int(df.duplicated().sum()))

df[["Time", "Amount"]].describe().T

In [ ]:
# =====================================================
# DISTRIBUCIÓN DE LA VARIABLE OBJETIVO
# =====================================================
conteo = df["Class"].value_counts().sort_index()
porcentaje = df["Class"].value_counts(normalize=True).sort_index() * 100

print(pd.DataFrame({"Transacciones": conteo, "%": porcentaje.round(3)})
      .rename(index={0: "Legítima (0)", 1: "Fraude (1)"}))

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=["Legítima", "Fraude"], y=conteo.values, palette=["#4C72B0", "#C44E52"], ax=ax)
ax.set_yscale("log")
ax.set_ylabel("Número de transacciones (escala log)")
ax.set_title("Desbalance de clases")
for i, v in enumerate(conteo.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
guardar_figura("01_desbalance_clases.png")
plt.show()

**Observación.** El fraude es extremadamente raro (alrededor del 0,17 % en el dataset original). Por eso el *accuracy* no sirve como métrica: un modelo que diga siempre "legítima" acertaría más del 99,8 % de las veces sin detectar un solo fraude.

In [ ]:
# =====================================================
# MONTO DE LA TRANSACCIÓN SEGÚN LA CLASE
# =====================================================
print(df.groupby("Class")["Amount"].describe().rename(index={0: "Legítima", 1: "Fraude"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=df, x=np.log1p(df["Amount"]), hue="Class", stat="density",
             common_norm=False, bins=60, ax=axes[0], palette=["#4C72B0", "#C44E52"])
axes[0].set_xlabel("log(1 + Amount)")
axes[0].set_title("Distribución del monto (escala log)")

sns.boxplot(data=df, x="Class", y="Amount", ax=axes[1], palette=["#4C72B0", "#C44E52"], showfliers=False)
axes[1].set_xticklabels(["Legítima", "Fraude"])
axes[1].set_title("Monto por clase (sin outliers)")
guardar_figura("02_monto_por_clase.png")
plt.show()

In [ ]:
# =====================================================
# PATRÓN TEMPORAL
# =====================================================
# Time está en segundos desde la primera transacción; se convierte a "hora del ciclo de 24 h"
hora = (df["Time"] // 3600) % 24
tasa_por_hora = df.groupby(hora)["Class"].mean() * 100
volumen_por_hora = df.groupby(hora).size()

fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.bar(volumen_por_hora.index, volumen_por_hora.values, color="#4C72B0", alpha=0.4, label="Volumen")
ax1.set_xlabel("Hora (relativa al inicio del registro)")
ax1.set_ylabel("Transacciones")
ax2 = ax1.twinx()
ax2.plot(tasa_por_hora.index, tasa_por_hora.values, color="#C44E52", marker="o", label="% fraude")
ax2.set_ylabel("% de fraude")
ax1.set_title("Volumen de transacciones y tasa de fraude por hora")
fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.88))
guardar_figura("03_patron_temporal.png")
plt.show()

**Observación.** En las horas de menor actividad (madrugada) el volumen cae, pero la proporción de fraude suele subir. Esto justifica crear una variable de hora del día en la fase de preparación.

In [ ]:
# =====================================================
# CORRELACIÓN DE CADA VARIABLE CON EL FRAUDE
# =====================================================
correlaciones = df.drop(columns="Class").corrwith(df["Class"]).sort_values()

plt.figure(figsize=(12, 4))
colores = ["#C44E52" if c < 0 else "#4C72B0" for c in correlaciones]
correlaciones.plot(kind="bar", color=colores)
plt.title("Correlación de cada variable con Class")
plt.ylabel("Correlación de Pearson")
guardar_figura("04_correlacion_con_clase.png")
plt.show()

print("Variables más relacionadas con el fraude:")
print(correlaciones.abs().sort_values(ascending=False).head(8).round(3))

---
# Fase 3 · Preparación de los datos

Pasos:
1. Eliminar duplicados (antes de dividir, para evitar fuga de información).
2. *Feature engineering*: hora del día en forma cíclica (seno/coseno) y logaritmo del monto.
3. División estratificada **60 % entrenamiento / 20 % validación / 20 % prueba**.
4. Escalado con `StandardScaler` ajustado **solo** con entrenamiento.
5. SMOTE **solo** sobre entrenamiento.

In [ ]:
# =====================================================
# LIMPIEZA Y FEATURE ENGINEERING
# =====================================================
filas_antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplicados eliminados: {filas_antes - len(df):,}")

def crear_variables(data):
    # Crea las variables derivadas. Se reutiliza tal cual en producción (Fase 6).
    data = data.copy()
    hora = (data["Time"] // 3600) % 24
    data["Hour_sin"] = np.sin(2 * np.pi * hora / 24)   # la hora 23 queda "cerca" de la hora 0
    data["Hour_cos"] = np.cos(2 * np.pi * hora / 24)
    data["Amount_log"] = np.log1p(data["Amount"])      # reduce la asimetría del monto
    return data

df = crear_variables(df)

FEATURES = [f"V{i}" for i in range(1, 29)] + ["Amount_log", "Hour_sin", "Hour_cos"]
TARGET = "Class"
print("Número de variables predictoras:", len(FEATURES))

In [ ]:
# =====================================================
# DIVISIÓN ENTRENAMIENTO / VALIDACIÓN / PRUEBA
# =====================================================
X = df[FEATURES]
y = df[TARGET]

# 80 % temporal, 20 % prueba
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
# Del 80 % temporal: 75 % entrenamiento, 25 % validación  ->  60 / 20 / 20
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=RANDOM_STATE
)

for nombre, yy in [("Entrenamiento", y_train), ("Validación", y_val), ("Prueba", y_test)]:
    print(f"{nombre:<14} {len(yy):>8,} filas | fraudes: {int(yy.sum()):>4} ({yy.mean()*100:.3f} %)")

In [ ]:
# =====================================================
# ESCALADO (ajustado solo con entrenamiento)
# =====================================================
scaler = StandardScaler().fit(X_train)

def escalar(data):
    return pd.DataFrame(scaler.transform(data[FEATURES]), columns=FEATURES, index=data.index)

X_train_s = escalar(X_train)
X_val_s = escalar(X_val)
X_test_s = escalar(X_test)

In [ ]:
# =====================================================
# BALANCEO DE CLASES CON SMOTE (solo entrenamiento)
# =====================================================
# sampling_strategy=0.1 -> los fraudes pasan a ser el 10 % de las legítimas.
# Se evita llegar a 50/50 para no generar demasiados ejemplos sintéticos.
smote = SMOTE(sampling_strategy=0.1, random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train_s, y_train)

resumen = pd.DataFrame({
    "Antes de SMOTE": y_train.value_counts().sort_index(),
    "Después de SMOTE": pd.Series(y_train_sm).value_counts().sort_index()
}).rename(index={0: "Legítima", 1: "Fraude"})
print(resumen)

resumen.T.plot(kind="bar", figsize=(7, 4), color=["#4C72B0", "#C44E52"], logy=True, rot=0)
plt.title("Conjunto de entrenamiento antes y después de SMOTE")
plt.ylabel("Transacciones (escala log)")
guardar_figura("05_smote.png")
plt.show()

**¿Por qué SMOTE solo en entrenamiento?** SMOTE crea fraudes *sintéticos* interpolando entre fraudes reales. Si se aplicara antes de dividir, versiones sintéticas de los fraudes de prueba aparecerían en entrenamiento y las métricas saldrían infladas. Validación y prueba conservan la proporción real, que es la que el modelo encontrará en producción.

---
# Fase 4 · Modelado

Todos los modelos se evalúan con las mismas funciones. Para cada uno se guardan las probabilidades en validación (para elegir el umbral) y en prueba (para la evaluación final).

In [ ]:
# =====================================================
# FUNCIONES DE APOYO
# =====================================================
resultados = {}   # nombre_modelo -> {"val": puntajes_validación, "test": puntajes_prueba}

def mejor_umbral(y_true, puntaje):
    # Umbral que maximiza el F1 en el conjunto recibido (se usa con VALIDACIÓN)
    p, r, t = precision_recall_curve(y_true, puntaje)
    f1 = 2 * p * r / (p + r + 1e-12)
    return float(t[np.nanargmax(f1[:-1])])

def calcular_metricas(y_true, puntaje, umbral):
    pred = (puntaje >= umbral).astype(int)
    return {
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred),
        "F1": f1_score(y_true, pred),
        "ROC-AUC": roc_auc_score(y_true, puntaje),
        "PR-AUC": average_precision_score(y_true, puntaje),
    }

def registrar(nombre, puntaje_val, puntaje_test):
    resultados[nombre] = {"val": np.asarray(puntaje_val), "test": np.asarray(puntaje_test)}
    m = calcular_metricas(y_val, puntaje_val, mejor_umbral(y_val, puntaje_val))
    print(f"{nombre} -> validación | PR-AUC={m['PR-AUC']:.4f}  ROC-AUC={m['ROC-AUC']:.4f}  F1={m['F1']:.4f}")

## 4.1 Regresión Logística (modelo de referencia)
Sirve como línea base: cualquier modelo más complejo debe superarla para justificar su uso.

In [ ]:
log_reg = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)
log_reg.fit(X_train_s, y_train)
registrar("Regresión Logística",
          log_reg.predict_proba(X_val_s)[:, 1],
          log_reg.predict_proba(X_test_s)[:, 1])

## 4.2 Random Forest + SMOTE (Bagging)
Entrena muchos árboles sobre muestras *bootstrap* y promedia sus votos, lo que reduce la varianza.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
    random_state=RANDOM_STATE
)
rf.fit(X_train_sm, y_train_sm)
registrar("Random Forest + SMOTE",
          rf.predict_proba(X_val_s)[:, 1],
          rf.predict_proba(X_test_s)[:, 1])

## 4.3 XGBoost con `scale_pos_weight` (Boosting)
Construye árboles en secuencia, cada uno corrigiendo los errores del anterior. El desbalance se maneja dándole más peso a los fraudes. Se usa *early stopping* con el conjunto de **validación** (nunca con prueba).

In [ ]:
peso_fraude = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight calculado: {peso_fraude:.1f}")

xgb = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    scale_pos_weight=peso_fraude,
    eval_metric="aucpr",
    early_stopping_rounds=50,
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_STATE
)
xgb.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)], verbose=100)
print("Mejor iteración:", xgb.best_iteration)

registrar("XGBoost (peso de clase)",
          xgb.predict_proba(X_val_s)[:, 1],
          xgb.predict_proba(X_test_s)[:, 1])

## 4.4 XGBoost + SMOTE
Mismo algoritmo, pero manejando el desbalance con SMOTE en lugar de pesos. Permite comparar las dos estrategias de balanceo.

In [ ]:
xgb_sm = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    eval_metric="aucpr",
    early_stopping_rounds=50,
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_STATE
)
xgb_sm.fit(X_train_sm, y_train_sm, eval_set=[(X_val_s, y_val)], verbose=100)

registrar("XGBoost + SMOTE",
          xgb_sm.predict_proba(X_val_s)[:, 1],
          xgb_sm.predict_proba(X_test_s)[:, 1])

## 4.5 Stacking Ensemble (mejora del ejemplo original)
Se conserva la idea del ejemplo: Random Forest y XGBoost como modelos base y Regresión Logística como meta-modelo. Cambios:
- Los modelos base manejan el desbalance (`class_weight` y `scale_pos_weight`).
- Validación cruzada **estratificada**, para que cada pliegue tenga fraudes.
- El meta-modelo combina **probabilidades** (`stack_method="predict_proba"`).

*Este paso puede tardar algunos minutos.*

In [ ]:
stacking = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(
            n_estimators=100, min_samples_leaf=2, class_weight="balanced_subsample",
            n_jobs=-1, random_state=RANDOM_STATE)),
        ("xgb", XGBClassifier(
            n_estimators=xgb.best_iteration + 1, learning_rate=0.05, max_depth=5,
            subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
            scale_pos_weight=peso_fraude, tree_method="hist",
            n_jobs=-1, random_state=RANDOM_STATE)),
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    stack_method="predict_proba",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=1
)
stacking.fit(X_train_s, y_train)
registrar("Stacking (RF + XGB)",
          stacking.predict_proba(X_val_s)[:, 1],
          stacking.predict_proba(X_test_s)[:, 1])

## 4.6 Red Neuronal Profunda (DNN)
Tres capas ocultas (64-32-16) con *Batch Normalization* y *Dropout* para evitar sobreajuste. Se entrena con los datos balanceados por SMOTE y se detiene cuando el PR-AUC de validación deja de mejorar.

In [ ]:
def construir_dnn(n_variables):
    modelo = keras.Sequential([
        keras.Input(shape=(n_variables,)),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(curve="PR", name="pr_auc"), keras.metrics.Recall(name="recall")]
    )
    return modelo

dnn = construir_dnn(len(FEATURES))
dnn.summary()

parada_temprana = keras.callbacks.EarlyStopping(
    monitor="val_pr_auc", mode="max", patience=6, restore_best_weights=True
)
historial_dnn = dnn.fit(
    X_train_sm.values, y_train_sm.values,
    validation_data=(X_val_s.values, y_val.values),
    epochs=60, batch_size=2048,
    callbacks=[parada_temprana], verbose=2
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(historial_dnn.history["loss"], label="Entrenamiento")
axes[0].plot(historial_dnn.history["val_loss"], label="Validación")
axes[0].set_title("DNN – Pérdida"); axes[0].set_xlabel("Época"); axes[0].legend()
axes[1].plot(historial_dnn.history["pr_auc"], label="Entrenamiento")
axes[1].plot(historial_dnn.history["val_pr_auc"], label="Validación")
axes[1].set_title("DNN – PR-AUC"); axes[1].set_xlabel("Época"); axes[1].legend()
guardar_figura("06_entrenamiento_dnn.png")
plt.show()

registrar("Red Neuronal (DNN)",
          dnn.predict(X_val_s.values, verbose=0).ravel(),
          dnn.predict(X_test_s.values, verbose=0).ravel())

## 4.7 Autoencoder para detección de anomalías
Enfoque **no supervisado**: el autoencoder se entrena **solo con transacciones legítimas** para que aprenda a reconstruirlas. Cuando recibe un fraude, que "no se parece" a lo que aprendió, lo reconstruye mal. El **error de reconstrucción** se usa como puntaje de anomalía: a mayor error, mayor sospecha.

Su ventaja es que puede detectar patrones de fraude nuevos que no estaban etiquetados en los datos de entrenamiento.

In [ ]:
X_train_normal = X_train_s[y_train == 0].values
X_val_normal = X_val_s[y_val == 0].values
n = len(FEATURES)

autoencoder = keras.Sequential([
    keras.Input(shape=(n,)),
    layers.Dense(20, activation="relu"),
    layers.Dense(10, activation="relu"),   # cuello de botella
    layers.Dense(20, activation="relu"),
    layers.Dense(n, activation="linear"),
])
autoencoder.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")

historial_ae = autoencoder.fit(
    X_train_normal, X_train_normal,
    validation_data=(X_val_normal, X_val_normal),
    epochs=60, batch_size=1024,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)],
    verbose=2
)

def error_reconstruccion(modelo, datos):
    reconstruido = modelo.predict(datos, verbose=0)
    return np.mean(np.square(datos - reconstruido), axis=1)

error_val = error_reconstruccion(autoencoder, X_val_s.values)
error_test = error_reconstruccion(autoencoder, X_test_s.values)
registrar("Autoencoder (anomalías)", error_val, error_test)

In [ ]:
umbral_ae = mejor_umbral(y_val, error_val)

plt.figure(figsize=(10, 4))
sns.histplot(x=np.log10(error_test + 1e-9), hue=y_test.map({0: "Legítima", 1: "Fraude"}).values,
             stat="density", common_norm=False, bins=80, palette=["#4C72B0", "#C44E52"])
plt.axvline(np.log10(umbral_ae), color="black", linestyle="--", label=f"Umbral = {umbral_ae:.3f}")
plt.xlabel("log10(error de reconstrucción)")
plt.title("Autoencoder – error de reconstrucción en prueba")
plt.legend()
guardar_figura("07_autoencoder_error.png")
plt.show()

Si el autoencoder funciona bien, la distribución roja (fraudes) aparece desplazada a la derecha: los fraudes se reconstruyen con más error que las transacciones legítimas.

---
# Fase 5 · Evaluación

Procedimiento:
1. Para cada modelo se elige el umbral que maximiza el F1 **en validación**.
2. Con ese umbral fijo se calculan las métricas **en prueba**, datos que ningún modelo vio.
3. El mejor modelo se selecciona por **PR-AUC en validación**, para no "elegir mirando el examen".

In [ ]:
# =====================================================
# TABLA COMPARATIVA EN EL CONJUNTO DE PRUEBA
# =====================================================
umbrales = {}
filas = []
for nombre, r in resultados.items():
    umbral = mejor_umbral(y_val, r["val"])
    umbrales[nombre] = umbral
    m = calcular_metricas(y_test, r["test"], umbral)
    m["PR-AUC (val)"] = average_precision_score(y_val, r["val"])
    filas.append({"Modelo": nombre, **m})

tabla = (pd.DataFrame(filas).set_index("Modelo")
         .sort_values("PR-AUC (val)", ascending=False))
tabla.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
tabla[["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]].plot(kind="bar", ax=ax, rot=20)
ax.set_ylim(0, 1.05)
ax.set_title("Comparación de modelos en el conjunto de prueba")
ax.legend(loc="lower right")
guardar_figura("08_comparacion_modelos.png")
plt.show()

In [ ]:
# =====================================================
# CURVAS ROC Y PRECISION-RECALL
# =====================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for nombre, r in resultados.items():
    fpr, tpr, _ = roc_curve(y_test, r["test"])
    axes[0].plot(fpr, tpr, label=f"{nombre} ({roc_auc_score(y_test, r['test']):.3f})")
    p, rc, _ = precision_recall_curve(y_test, r["test"])
    axes[1].plot(rc, p, label=f"{nombre} ({average_precision_score(y_test, r['test']):.3f})")

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[0].set(title="Curva ROC", xlabel="Tasa de falsos positivos", ylabel="Tasa de verdaderos positivos")
axes[1].axhline(y_test.mean(), color="k", linestyle="--", alpha=0.5, label="Azar")
axes[1].set(title="Curva Precision-Recall", xlabel="Recall", ylabel="Precision")
for ax in axes:
    ax.legend(fontsize=8)
guardar_figura("09_curvas_roc_pr.png")
plt.show()

**¿Por qué mirar la curva Precision-Recall?** Con un desbalance tan grande, la curva ROC puede verse casi perfecta aunque el modelo genere muchas falsas alarmas, porque la tasa de falsos positivos se divide entre cientos de miles de transacciones legítimas. La curva PR se concentra en la clase fraude y muestra diferencias reales entre modelos.

In [ ]:
# =====================================================
# MATRICES DE CONFUSIÓN
# =====================================================
n_modelos = len(resultados)
columnas = 4
filas_fig = int(np.ceil(n_modelos / columnas))
fig, axes = plt.subplots(filas_fig, columnas, figsize=(4.2 * columnas, 3.8 * filas_fig))
axes = np.array(axes).ravel()

for ax, (nombre, r) in zip(axes, resultados.items()):
    pred = (r["test"] >= umbrales[nombre]).astype(int)
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, display_labels=["Legítima", "Fraude"],
        cmap="Blues", colorbar=False, ax=ax
    )
    ax.set_title(nombre, fontsize=10)
for ax in axes[n_modelos:]:
    ax.axis("off")
plt.tight_layout()
guardar_figura("10_matrices_confusion.png")
plt.show()

In [ ]:
# =====================================================
# ANÁLISIS DE COSTO PARA EL NEGOCIO
# =====================================================
# Supuesto (ajústalo y justifícalo en el documento):
#   - Fraude no detectado (FN): se pierde el monto de la transacción.
#   - Falsa alarma (FP): costo fijo de revisión y atención al cliente.
COSTO_REVISION = 5.0   # euros por falsa alarma

montos_test = df.loc[X_test.index, "Amount"].values
y_test_arr = y_test.values

def costo_modelo(pred):
    fn = (y_test_arr == 1) & (pred == 0)
    fp = (y_test_arr == 0) & (pred == 1)
    tp = (y_test_arr == 1) & (pred == 1)
    return {
        "Fraude evitado (€)": montos_test[tp].sum(),
        "Fraude perdido (€)": montos_test[fn].sum(),
        "Falsas alarmas": int(fp.sum()),
        "Costo total (€)": montos_test[fn].sum() + fp.sum() * COSTO_REVISION,
    }

costos = {"Sin modelo": costo_modelo(np.zeros_like(y_test_arr))}
for nombre, r in resultados.items():
    costos[nombre] = costo_modelo((r["test"] >= umbrales[nombre]).astype(int))

tabla_costos = pd.DataFrame(costos).T
tabla_costos["Ahorro vs. sin modelo (€)"] = (
    tabla_costos.loc["Sin modelo", "Costo total (€)"] - tabla_costos["Costo total (€)"]
)
tabla_costos.sort_values("Costo total (€)").round(2)

In [ ]:
# =====================================================
# SELECCIÓN DEL MEJOR MODELO (por PR-AUC en validación)
# =====================================================
MEJOR_MODELO = tabla.index[0]
print(f"Mejor modelo: {MEJOR_MODELO}")
print(f"Umbral elegido en validación: {umbrales[MEJOR_MODELO]:.4f}\n")

pred_mejor = (resultados[MEJOR_MODELO]["test"] >= umbrales[MEJOR_MODELO]).astype(int)
print(classification_report(y_test, pred_mejor, target_names=["Legítima", "Fraude"], digits=4))

# Verificación de los criterios de éxito definidos en la Fase 1
m = tabla.loc[MEJOR_MODELO]
criterios = {
    "PR-AUC >= 0.80": m["PR-AUC"] >= 0.80,
    "Recall >= 0.80": m["Recall"] >= 0.80,
    "Precision >= 0.70": m["Precision"] >= 0.70,
    "Ahorro positivo": tabla_costos.loc[MEJOR_MODELO, "Ahorro vs. sin modelo (€)"] > 0,
}
for criterio, cumple in criterios.items():
    print(f"{'✔' if cumple else '✘'} {criterio}")

## Interpretabilidad del modelo
Se usa **SHAP** sobre XGBoost para explicar qué variables empujan una predicción hacia "fraude". A diferencia del `plot_importance` del ejemplo original, que por defecto solo cuenta cuántas veces se usa una variable en los árboles, SHAP mide el aporte real de cada variable y su dirección.

In [ ]:
try:
    import shap
    muestra = X_test_s.sample(n=min(2000, len(X_test_s)), random_state=RANDOM_STATE)
    explicador = shap.TreeExplainer(xgb)
    valores_shap = explicador.shap_values(muestra)

    shap.summary_plot(valores_shap, muestra, show=False, max_display=15)
    plt.title("SHAP – impacto de cada variable (XGBoost)")
    guardar_figura("11_shap_resumen.png")
    plt.show()

    shap.summary_plot(valores_shap, muestra, plot_type="bar", show=False, max_display=15)
    plt.title("SHAP – importancia media absoluta")
    guardar_figura("12_shap_importancia.png")
    plt.show()
except ImportError:
    print("SHAP no está instalado. Ejecuta: pip install shap")

**Cómo leer el gráfico de SHAP:** cada punto es una transacción. Los puntos a la derecha empujan la predicción hacia *fraude*; el color indica si el valor de la variable es alto (rojo) o bajo (azul). Por ejemplo, si los puntos azules de `V14` aparecen a la derecha, valores bajos de `V14` están asociados a fraude.

---
# Fase 6 · Despliegue

Se guardan todos los artefactos necesarios para que la aplicación web (dashboard, alertas y visualización de transacciones sospechosas) haga **exactamente** el mismo preprocesamiento que el entrenamiento.

In [ ]:
# =====================================================
# GUARDAR ARTEFACTOS
# =====================================================
joblib.dump(scaler, os.path.join(MODELS_DIR, "scaler.joblib"))
joblib.dump(log_reg, os.path.join(MODELS_DIR, "regresion_logistica.joblib"))
joblib.dump(rf, os.path.join(MODELS_DIR, "random_forest.joblib"))
joblib.dump(xgb, os.path.join(MODELS_DIR, "xgboost.joblib"))
joblib.dump(xgb_sm, os.path.join(MODELS_DIR, "xgboost_smote.joblib"))
joblib.dump(stacking, os.path.join(MODELS_DIR, "stacking.joblib"))
dnn.save(os.path.join(MODELS_DIR, "dnn.keras"))
autoencoder.save(os.path.join(MODELS_DIR, "autoencoder.keras"))

ARCHIVOS = {
    "Regresión Logística": "regresion_logistica.joblib",
    "Random Forest + SMOTE": "random_forest.joblib",
    "XGBoost (peso de clase)": "xgboost.joblib",
    "XGBoost + SMOTE": "xgboost_smote.joblib",
    "Stacking (RF + XGB)": "stacking.joblib",
    "Red Neuronal (DNN)": "dnn.keras",
    "Autoencoder (anomalías)": "autoencoder.keras",
}

config = {
    "features": FEATURES,
    "mejor_modelo": MEJOR_MODELO,
    "umbrales": umbrales,
    "archivos": ARCHIVOS,
    "metricas_prueba": tabla.round(4).to_dict(orient="index"),
    "costo_revision": COSTO_REVISION,
    "fecha_entrenamiento": datetime.now().isoformat(timespec="seconds"),
}
with open(os.path.join(MODELS_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Artefactos guardados en", MODELS_DIR)
print(sorted(os.listdir(MODELS_DIR)))

In [ ]:
# =====================================================
# FUNCIÓN DE PREDICCIÓN PARA PRODUCCIÓN
# =====================================================
# Recibe transacciones con las columnas originales (Time, V1..V28, Amount),
# carga los artefactos desde disco y aplica el mismo preprocesamiento del entrenamiento.

def cargar_modelo(nombre, config):
    ruta = os.path.join(MODELS_DIR, config["archivos"][nombre])
    return keras.models.load_model(ruta) if ruta.endswith(".keras") else joblib.load(ruta)

def nivel_riesgo(puntaje, umbral):
    if puntaje >= umbral:
        return "ALTO"
    if puntaje >= umbral * 0.5:
        return "MEDIO"
    return "BAJO"

def predecir_transacciones(df_crudo, nombre_modelo=None):
    with open(os.path.join(MODELS_DIR, "config.json"), encoding="utf-8") as f:
        cfg = json.load(f)
    nombre_modelo = nombre_modelo or cfg["mejor_modelo"]
    escalador = joblib.load(os.path.join(MODELS_DIR, "scaler.joblib"))
    modelo = cargar_modelo(nombre_modelo, cfg)
    umbral = cfg["umbrales"][nombre_modelo]

    datos = crear_variables(df_crudo)[cfg["features"]]
    datos_s = escalador.transform(datos)

    if nombre_modelo == "Autoencoder (anomalías)":
        puntaje = error_reconstruccion(modelo, datos_s)
    elif nombre_modelo == "Red Neuronal (DNN)":
        puntaje = modelo.predict(datos_s, verbose=0).ravel()
    else:
        puntaje = modelo.predict_proba(pd.DataFrame(datos_s, columns=cfg["features"]))[:, 1]

    salida = df_crudo[["Time", "Amount"]].copy()
    salida["puntaje"] = puntaje
    salida["alerta"] = puntaje >= umbral
    salida["riesgo"] = [nivel_riesgo(p, umbral) for p in puntaje]
    return salida

# Prueba con algunas transacciones de prueba (en formato crudo)
indices_demo = list(y_test[y_test == 1].index[:3]) + list(y_test[y_test == 0].index[:3])
demo = df.loc[indices_demo, ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount"]]
resultado_demo = predecir_transacciones(demo)
resultado_demo["Clase real"] = df.loc[indices_demo, "Class"].map({0: "Legítima", 1: "Fraude"}).values
resultado_demo

In [ ]:
# =====================================================
# SIMULACIÓN DE ALERTAS EN TIEMPO REAL
# =====================================================
# Se "reproduce" un flujo de transacciones del conjunto de prueba como si llegaran
# una a una. Es la misma lógica que usará el monitoreo del dashboard.
columnas_crudas = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount"]
# Para la demostración se mezclan 20 fraudes con 480 transacciones legítimas del conjunto de
# prueba (con la proporción real, 500 transacciones tendrían apenas 1 fraude).
prueba = df.loc[X_test.index]
fraudes_demo = prueba[prueba["Class"] == 1].sample(n=min(20, int(y_test.sum())), random_state=7)
legitimas_demo = prueba[prueba["Class"] == 0].sample(n=480, random_state=7)
flujo = pd.concat([fraudes_demo, legitimas_demo]).sort_values("Time")

puntuado = predecir_transacciones(flujo[columnas_crudas])
inicio = datetime(2013, 9, 1)

print(f"Monitoreando {len(flujo)} transacciones con el modelo: {MEJOR_MODELO}\n")
for idx, fila in puntuado.iterrows():
    if fila["alerta"]:
        momento = inicio + timedelta(seconds=float(fila["Time"]))
        real = "FRAUDE REAL" if flujo.loc[idx, "Class"] == 1 else "falsa alarma"
        print(f"🚨 [{momento:%d/%m %H:%M:%S}] Transacción #{idx} | Monto: €{fila['Amount']:>8.2f} "
              f"| Puntaje: {fila['puntaje']:.3f} | Riesgo {fila['riesgo']} ({real})")

alertas = puntuado["alerta"].sum()
fraudes_reales = int(flujo["Class"].sum())
detectados = int((puntuado["alerta"] & (flujo["Class"] == 1)).sum())
print(f"\nResumen: {alertas} alertas | {detectados} de {fraudes_reales} fraudes reales detectados")

### Próximo paso: aplicación web
Con los archivos de la carpeta `models/` se construye la aplicación (por ejemplo, con **Streamlit**), que debe incluir:

- **Dashboard de fraude:** KPIs (transacciones, alertas, monto en riesgo), tabla comparativa de modelos y gráficos de esta fase.
- **Alertas en tiempo real:** la lógica de la celda anterior, mostrando cada transacción a medida que "llega".
- **Visualización de transacciones sospechosas:** tabla filtrable con puntaje, nivel de riesgo, monto y hora, generada con `predecir_transacciones()`.

### Conclusiones
Redacta aquí las conclusiones con base en los resultados obtenidos: qué modelo fue el mejor y por qué, si se cumplieron los criterios de éxito de la Fase 1, qué estrategia de balanceo funcionó mejor (SMOTE vs. pesos de clase), qué aportó el autoencoder como enfoque no supervisado y cuál es el ahorro económico estimado.